# CSE 151B — Development notebook

Live in **`notebooks/dev.ipynb`**. Imports resolve **`REPO_ROOT`** automatically so `data/` and `results/` work whether the Jupyter cwd is the project root or `notebooks/`.

This notebook follows **`starter_code_cse151b_comp.ipynb`** end-to-end (vLLM inference → `Judger` scoring → JSONL output), but runs on a **held-out slice** of `public.jsonl` so you can iterate faster.

1. (**Colab**) Copy `public.jsonl` from Drive into `data/` (skip locally if the file already exists).
2. (**Colab**) Copy frozen eval from Drive (`data/dev.jsonl` or `data/eval/holdout.jsonl`) — no resampling in-notebook.
3. Same prompts, model load, generation, and scoring as the starter. **dev-010-bf:** `BUDGET_FORCING=True` (§7b) — s1-style `Wait` on early `</think>`. **§1.3 smoke:** `SMOKE_TEST=True` on 20 multi-blank ids before full holdout.
4. Writes **`results/dev_results_{variant}_{Nk}[_bf][_php][_scK][_smoke].jsonl`** (+ `.responses.jsonl` checkpoint; optional `.sc_traces.jsonl` when SC on)..
5. §10 prints metrics and a registry line for [`docs/log/experiments.md`](../docs/log/experiments.md).

Use the starter notebook for **full** public evaluation; use **`notebooks/submission.ipynb`** for private CSV export.

## 1. Environment (same as starter)

**Colab (A100):** run the `%pip` cell below, restart runtime, continue. **Local:** use `uv` / your venv as in the starter — install the same packages (`vllm`, `transformers`, …).

> **Note:** `bitsandbytes` is not needed — Qwen3-4B fits in native bf16 on A100 (~8 GB), which is faster than quantized loading.

In [1]:
# # Colab: skip when working locally with an existing venv.
%pip install -q sympy numpy tqdm "antlr4-python3-runtime==4.11.1"
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu129 --upgrade
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.20.0/vllm-0.20.0+cu129-cp38-abi3-manylinux_2_31_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip install -q transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 15.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 MB 80.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 82.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 248.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 129.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 245.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 234.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 182.1 MB/s eta 0:00:0000:0100:01
     ━━━

## 2. Imports & configuration

Matches the starter notebook; dev-specific keys:

- `DATA_PATH` — set in §4 from Drive or repo (`data/dev.jsonl` or `data/eval/holdout.jsonl`)
- `MAX_TOKENS` — generation cap (8192 = pub-001; 16384 = dev-007/dev-008; 32768 = dev-009 ablation, rejected)
- `PROMPT_VARIANT` — `"baseline"` | `"concise"` | `"multi_blank"` (roadmap §1.3) | `"verify_prompt"` (dev-013) | `"precision"` (dev-014) | `"precision_v2"` (dev-015, rejected — flat) | `"precision_v3"` (dev-016 — slimmed to format + domain-decimal only)
- `USE_XML_WRAPPERS` — system: `<instructions>` + `<constraints>` with one `<rule>` per constraint; user: `<problem>`; tags `_xml` in output filename
- `SMOKE_TEST` / `SMOKE_N` — §1.3 Colab smoke: multi-blank free-form only (default 20 rows)
- `BUDGET_FORCING` — s1-style `Wait` injection on early `</think>` ([survey §A](../../docs/research/2026-05-24-improvement-techniques-survey.md))
- `BUDGET_FORCE_MAX_ITER` / `BUDGET_FORCE_WAIT` / `BUDGET_FORCE_MIN_THINK_CHARS` (FF) / `BUDGET_FORCE_MIN_THINK_CHARS_MCQ` (guard — default disables MCQ injection) / `BUDGET_FORCE_CONTEXT_CAP`
- `PHP_ENABLED` / `PHP_FF_ONLY` / `PHP_MAX_TOKENS` — progressive-hint pass-2 on FF (§7c / §8b)
- `SELF_CONSISTENCY` / `SC_K` / `SC_ANCHOR_PATH` — K-sample vote (§7d); anchor = sample 0 or external jsonl
- `RUN_ID` — optional registry id (e.g. `dev-010-bf`); auto-derived when `None`
- `OUTPUT_PATH` — `results/dev_results_{variant}_{Nk}[_xml][_bf][_php][_scK][_smoke].jsonl`


In [17]:
import json
import os
from pathlib import Path


def _repo_root() -> Path:
    """Project root (parent of `data/`), whether cwd is repo root or `notebooks/`."""
    here = Path.cwd().resolve()
    if (here / "data").is_dir():
        return here
    if here.name == "notebooks" and (here.parent / "data").is_dir():
        return here.parent
    up = here.parent
    if (up / "data").is_dir():
        return up
    return here


REPO_ROOT = _repo_root()

# ── Eval set (frozen on Drive — copied in §4, never rebuilt here) ─────────────
DATA_PATH = None  # resolved in §4

# ── Experiment knobs ──────────────────────────────────────────────────────────
# PROMPT_VARIANT controls which system prompt set is used.
#   "baseline"        — original prompts (matches pub-001 / 8k run)
#   "concise"         — thinking-efficiency prompts (roadmap §1.2)
#   "multi_blank"     — separate \boxed{} per blank (roadmap §1.3)
#   "verify_prompt"  — verification forcing + multi_blank format (dev-013)
#   "precision"        — exact-form / no-round + multi_blank (grader 1e-8 fix)
#   "precision_v2"     — precision + 6 fixes from dev-014 item-level error analysis (rejected: flat)
#   "precision_v3"     — precision_v2 slimmed: format clauses + domain-decimal only
#   "precision_xml"    — structural tagging: <instructions>/<constraints>/<response_format> system,
#                        <problem><statement><blanks|options><output_format> user (dev-xml-001)
#
# dev-013 verification-forcing ablation (compare vs dev-008 / pub-002):
#   PROMPT_VARIANT = "verify_prompt"
#   SELF_CONSISTENCY = False
#   PHP_ENABLED = False
#   BUDGET_FORCING = False
#
# §1.3 smoke order (Colab): run baseline first, then multi_blank (same 20 ids).
# dev-010-bf ablation: use "baseline" at 16k to compare vs pub-002 failure mode.
PROMPT_VARIANT = "precision_xml_verify"

# Wrap prompts in XML when True (orthogonal to PROMPT_VARIANT):
#   system → <instructions>role</instructions> + <constraints><rule>…</rule>…</constraints>
#   user   → <problem>question (+ options)</problem>
USE_XML_WRAPPERS = False

# §1.3 Colab smoke: multi-blank free-form only (~20 items, ~15 min on A100)
SMOKE_TEST = False
SMOKE_N    = 20

# max_tokens: 8192 (pub-001 / dev-003), 16384 (dev-007 / dev-008 / dev-010-bf), 32768 (dev-009 — rejected)
MAX_TOKENS = 16384

# ── Budget forcing (s1-style; survey §A) ─────────────────────────────────────
# On early </think>, suppress close and append " Wait" (max 2×). ~1.3× inference.
BUDGET_FORCING = False
BUDGET_FORCE_MAX_ITER = 2
BUDGET_FORCE_WAIT = " Wait"
BUDGET_FORCE_MIN_THINK_CHARS = 10**9   # FF: force if thinking body shorter at close attempt
BUDGET_FORCE_MIN_THINK_CHARS_MCQ = 0  # MCQ: effectively skip Wait injection (FF-biased)
BUDGET_FORCE_CONTEXT_CAP = 14 * 1024  # stop new passes above this prompt+gen tokens

# ── Progressive-Hint Prompting (PHP; survey §1.13) ───────────────────────────
# Two-pass on FF only: pass-1 produces tentative \boxed{}; pass-2 re-prompts
# with a neutral hint and uses pass-2 \boxed{} as final. ~2× inference on FF.
PHP_ENABLED = False
PHP_FF_ONLY = True            # skip MCQ (already 72% pre-PHP, low headroom)
PHP_MAX_TOKENS = MAX_TOKENS   # pass-2 budget; pass-1 uses MAX_TOKENS
PHP_SKIP_IF_NO_PASS1_ANSWER = True  # if pass-1 has no \boxed{}, keep pass-1 raw

# ── Self-consistency (SC; roadmap §1.4) ─────────────────────────────────────
# K independent samples per item → extract / canon / vote (§7d). ~K× inference.
SELF_CONSISTENCY = False
SC_K = 5
# Optional K=1 anchor traces (e.g. pub-002 responses). Missing ids use sample 0.
SC_ANCHOR_PATH = None  # e.g. REPO_ROOT / "data/full_public_16k.responses.jsonl"

RUN_ID = "dev-precision-prompt"  # auto: dev-010-bf, dev-010-bf-smoke, dev-008, dev-007, …

# ── Same as starter ───────────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID   = "0"

_TOKEN_K = MAX_TOKENS // 1024
if RUN_ID is None:
    if BUDGET_FORCING:
        RUN_ID = "dev-010-bf-smoke" if SMOKE_TEST else "dev-010-bf"
    elif SMOKE_TEST:
        if PROMPT_VARIANT == "multi_blank":
            RUN_ID = "dev-008-smoke"
        elif PROMPT_VARIANT == "baseline":
            RUN_ID = "dev-008-baseline-smoke"
        else:
            RUN_ID = f"dev-smoke-{PROMPT_VARIANT}"
    elif PROMPT_VARIANT == "verify_prompt":
        RUN_ID = "dev-013-verify"
    elif PROMPT_VARIANT == "multi_blank":
        RUN_ID = "dev-008"
    elif MAX_TOKENS >= 16384:
        RUN_ID = "dev-007"
    else:
        RUN_ID = "dev-003"
_smoke_tag = "_smoke" if SMOKE_TEST else ""
_xml_tag = "_xml" if USE_XML_WRAPPERS else ""
_bf_tag = "_bf" if BUDGET_FORCING else ""
_php_tag = "_php" if PHP_ENABLED else ""
_sc_tag = f"_sc{SC_K}" if SELF_CONSISTENCY else ""
if PHP_ENABLED and RUN_ID is not None and "php" not in RUN_ID:
    RUN_ID = f"{RUN_ID}-php"
if SELF_CONSISTENCY and RUN_ID is not None and "sc" not in RUN_ID:
    RUN_ID = f"{RUN_ID}-sc{SC_K}"
OUTPUT_PATH = str(
    REPO_ROOT
    / f"results/dev_results_{PROMPT_VARIANT}_{_TOKEN_K}k{_xml_tag}{_bf_tag}{_php_tag}{_sc_tag}{_smoke_tag}.jsonl"
)

print(f"REPO_ROOT      = {REPO_ROOT}")
print(f"RUN_ID         = {RUN_ID}")
print(f"MAX_TOKENS     = {MAX_TOKENS} ({_TOKEN_K}k)")
print(f"PROMPT_VARIANT = {PROMPT_VARIANT!r}  →  {OUTPUT_PATH}")
print(f"USE_XML_WRAPPERS = {USE_XML_WRAPPERS}")
print(f"BUDGET_FORCING = {BUDGET_FORCING}" + (
    f" (max_iter={BUDGET_FORCE_MAX_ITER}, wait={BUDGET_FORCE_WAIT!r}, "
    f"min_think_ff={BUDGET_FORCE_MIN_THINK_CHARS}, min_think_mcq={BUDGET_FORCE_MIN_THINK_CHARS_MCQ})"
    if BUDGET_FORCING else ""
))
print(f"PHP_ENABLED    = {PHP_ENABLED}" + (f" (ff_only={PHP_FF_ONLY}, pass2_max_tokens={PHP_MAX_TOKENS})" if PHP_ENABLED else ""))
print(f"SELF_CONSISTENCY = {SELF_CONSISTENCY}" + (f" (K={SC_K})" if SELF_CONSISTENCY else ""))
print(f"SMOKE_TEST     = {SMOKE_TEST}" + (f" (n={SMOKE_N} multi-blank FF)" if SMOKE_TEST else ""))

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import glob
import site

def _prepend_nvidia_libs_to_ld_path() -> None:
    roots = list(site.getsitepackages())
    user_site = getattr(site, "getusersitepackages", lambda: None)()
    if user_site:
        roots.append(user_site)
    libdirs: list[str] = []
    seen: set[str] = set()
    for root in roots:
        for d in glob.glob(os.path.join(root, "nvidia", "*", "lib")):
            if os.path.isdir(d) and d not in seen:
                seen.add(d)
                libdirs.append(d)
    if libdirs:
        sep = os.pathsep
        os.environ["LD_LIBRARY_PATH"] = sep.join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(sep)


_prepend_nvidia_libs_to_ld_path()

import contextlib
import io
import re
import sys
from typing import Optional

@contextlib.contextmanager
def _jupyter_stdout_for_vllm():
    """Jupyter often replaces stdout with a stream that has no fileno(); vLLM workers need a real FD. Use only around vLLM load/generate so notebook print() still works elsewhere."""
    try:
        sys.stdout.fileno()
    except (io.UnsupportedOperation, AttributeError, OSError):
        saved_out, saved_err = sys.stdout, sys.stderr
        dup1, dup2 = os.dup(1), os.dup(2)
        try:
            sys.stdout = os.fdopen(dup1, "w", buffering=1)
            sys.stderr = os.fdopen(dup2, "w", buffering=1)
            yield
        finally:
            sys.stdout.close()
            sys.stderr.close()
            sys.stdout, sys.stderr = saved_out, saved_err
    else:
        yield

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm


REPO_ROOT      = /content
RUN_ID         = dev-precision-prompt
MAX_TOKENS     = 16384 (16k)
PROMPT_VARIANT = 'precision_xml_verify'  →  /content/results/dev_results_precision_xml_verify_16k.jsonl
USE_XML_WRAPPERS = False
BUDGET_FORCING = False
PHP_ENABLED    = False
SELF_CONSISTENCY = False
SMOKE_TEST     = False


## 3. Colab: mount Drive

Run on **Google Colab** before §4. Locally, §4 uses `data/dev.jsonl` or `data/eval/holdout.jsonl` if already in the repo.

In [18]:
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    print("Skip: not Google Colab — §4 uses repo eval JSONL.")
    DRIVE_PROJECT_ROOT = None
else:
    drive.mount("/content/drive")
    DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CSE151B")
    print(f"DRIVE_PROJECT_ROOT = {DRIVE_PROJECT_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_PROJECT_ROOT = /content/drive/MyDrive/CSE151B


## 4. Copy frozen eval from Drive (no rebuild)

Uses the eval JSONL already on Drive. Prefers legacy **`data/dev.jsonl`** (112-row / dev-008 slice) if present, else **`data/eval/holdout.jsonl`** (225-row). Rebuild only via `python scripts/build_eval_holdout.py` outside this notebook.

In [19]:
import shutil

_EVAL_CANDIDATES = (
    "data/eval/holdout_20p.jsonl",
)


def _resolve_eval_path() -> Path:
    for rel in _EVAL_CANDIDATES:
        local = REPO_ROOT / rel
        try:
            drive_root = DRIVE_PROJECT_ROOT
        except NameError:
            drive_root = None
        if drive_root is not None:
            src = drive_root / rel
            if src.is_file():
                local.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, local)
                print(f"Copied from Drive: {src} → {local.resolve()}")
                return local.resolve()
        if local.is_file():
            print(f"Using local eval: {local.resolve()}")
            return local.resolve()
    raise FileNotFoundError(
        "No frozen eval JSONL found. On Drive upload one of:\n"
        + "\n".join(f"  CSE151B/{p}" for p in _EVAL_CANDIDATES)
        + "\nOr place the same path under the repo `data/` tree."
    )


DATA_PATH = str(_resolve_eval_path())

Copied from Drive: /content/drive/MyDrive/CSE151B/data/eval/holdout_20p.jsonl → /content/data/eval/holdout_20p.jsonl


## 5. Load dataset

Uses `DATA_PATH` from §4. When `SMOKE_TEST=True`, keeps only multi-blank free-form rows (first `SMOKE_N`).

In [20]:
def n_ans_placeholders(question: str) -> int:
    return question.count("[ANS]")


data = [json.loads(line) for line in open(DATA_PATH)]

if SMOKE_TEST:
    multi_blank = sorted(
        (
            d for d in data
            if not d.get("options") and n_ans_placeholders(d["question"]) > 1
        ),
        key=lambda d: d.get("id", 0),
    )
    data = multi_blank[:SMOKE_N]
    smoke_ids = [d["id"] for d in data]
    print(
        f"SMOKE §1.3: {len(data)} multi-blank free-form "
        f"(from {len(multi_blank)} in dev slice, cap SMOKE_N={SMOKE_N})"
    )
    print(f"  ids: {smoke_ids}")

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options") for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form) from {DATA_PATH}")

mcq_sample  = next((d for d in data if d.get("options")), None)
free_sample = next((d for d in data if not d.get("options")), None)
multi_sample = next(
    (d for d in data if not d.get("options") and n_ans_placeholders(d["question"]) > 1),
    free_sample,
)

if mcq_sample:
    print("\n── MCQ sample ──")
    print(json.dumps(mcq_sample, indent=2)[:1200], "...\n" if len(json.dumps(mcq_sample)) > 1200 else "\n")
if free_sample:
    print("── Free-form sample ──")
    print(json.dumps(free_sample, indent=2)[:1200], "...\n" if len(json.dumps(free_sample)) > 1200 else "\n")
if multi_sample and multi_sample is not free_sample:
    print(f"── Multi-blank sample ({n_ans_placeholders(multi_sample['question'])} blanks) ──")
    print(json.dumps(multi_sample, indent=2)[:1200], "...\n" if len(json.dumps(multi_sample)) > 1200 else "\n")

Loaded 225 questions  (75 MCQ, 150 free-form) from /content/data/eval/holdout_20p.jsonl

── MCQ sample ──
{
  "question": "An ordinary deck of cards containing 26 red cards and 26 black cards is shuffled and dealt out one card at a time without replacement. Let $X_i$ be the color of the $i$th card. Compute $H(X_1,X_2,\\ldots,X_{52})$ in bits.",
  "options": [
    "52",
    "48.8",
    "49.9",
    "50.0",
    "51.5",
    "46.5",
    "50.2",
    "47.3",
    "45.6",
    "53.2"
  ],
  "answer": "B",
  "id": 33
} 

── Free-form sample ──
{
  "question": "Reduce the fraction ${\\frac{25}{40}}$. [ANS]",
  "answer": [
    "5/8"
  ],
  "id": 3
} 

── Multi-blank sample (2 blanks) ──
{
  "question": "Let $p$ be the price of an item and $q$ be the number of items sold at that price, where $q=f(p)$. What do the following quantities mean in terms of prices and quantities sold? 1. $f(45)$ is the [ANS] A. price for which 45 items are sold  B. rate at which the price is changing when 45 items are sold

## 6. Prompt construction

Controlled by `PROMPT_VARIANT` in §2. With `USE_XML_WRAPPERS`, the system message is `<instructions>` (role/task) plus `<constraints>` containing one `<rule>` per format constraint; the user message is `<problem>`.

| Variant | Description | Motivation |
|---------|-------------|------------|
| `"baseline"` | Original pub-001 prompts | Reproducibility baseline |
| `"concise"` | Adds "non-repetitive, commit once identified" instructions | Roadmap §1.2 — rejected at 8k |
| `"multi_blank"` | Separate `\boxed{a}, \boxed{b}, ...` in blank order (no `Answer N:` labels) | Roadmap §1.3 — must match `judger.extract_all_boxed` contiguous-group rule |
| `"verify_prompt"` | Same multi-blank output format + verify before boxing (substitute back, units/range/signs, independent check) | dev-013 ablation vs `multi_blank` at 16k |

Switch variant → re-run this cell + §8 (generation). Results land in `results/dev_results_{PROMPT_VARIANT}_{Nk}[_xml][_smoke].jsonl`.

In [ ]:
# ── Baseline prompts (match pub-001 / 8k run) ────────────────────────────────
# Roles vs constraints split enables <instructions>/<constraints> when USE_XML_WRAPPERS.
_ROLE_SOLVE = "You are an expert mathematician. Solve the problem step-by-step."
_ROLE_SOLVE_THINK = "You are an expert mathematician. Think step-by-step to solve the problem."
_ROLE_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer."
)


def _compose_system(instructions: str, rules: str | list[str]) -> str:
    """Flat prose when USE_XML_WRAPPERS=False\\; <instructions> + <constraints>/<rule> when True."""
    if isinstance(rules, str):
        rules = [rules] if rules.strip() else []
    else:
        rules = [r.strip() for r in rules if r and str(r).strip()]
    instructions = instructions.strip()
    if not USE_XML_WRAPPERS:
        flat = " ".join(rules)
        return f"{instructions} {flat}" if flat else instructions
    parts = [f"<instructions>{instructions}</instructions>"]
    if rules:
        rule_tags = "\n".join(f"<rule>{r}</rule>" for r in rules)
        parts.append(f"<constraints>\n{rule_tags}\n</constraints>")
    return "\n\n".join(parts)


_MATH_BASELINE = _compose_system(
    _ROLE_SOLVE,
    [
        "Put your final answer inside \\boxed{}.",
        "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
        "e.g. \\boxed{3, 7}.",
    ],
)

_MCQ_BASELINE = _compose_system(
    _ROLE_MCQ,
    ["Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."],
)

# ── Concise prompts (roadmap §1.2 — thinking-efficiency experiment) ───────────
# Motivated by data/incorrect_mcq_sample.txt: 84% of wrong MCQ are truncated
# mid-think. Dominant patterns in traces:
#   1. Circular "Wait, no..." loops re-deriving already-correct steps
#   2. Repeated interpretation second-guessing on the same setup
#   3. Model never commits despite having the right answer
# Goal: shorten median think trace so more responses finish within 8k tokens.
_MATH_CONCISE = _compose_system(
    _ROLE_SOLVE_THINK,
    [
        "Keep your reasoning focused and non-repetitive: do not re-derive steps you have already "
        "completed, and avoid going in circles.",
        "Put your final answer inside \\boxed{}.",
        "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
        "e.g. \\boxed{3, 7}.",
    ],
)

_MCQ_CONCISE = _compose_system(
    _ROLE_SOLVE_THINK,
    [
        "Keep your reasoning focused and non-repetitive: do not re-derive steps you have already "
        "completed, and avoid going in circles.",
        "Once you have identified the best answer, commit to it immediately and output ONLY its "
        "letter inside \\boxed{}, e.g. \\boxed{C}.",
    ],
)

# ── Multi-blank prompts (roadmap §1.3) ────────────────────────────────────────
# judger.extract_all_boxed groups contiguous \\boxed{} only when gap is whitespace/punct.
# Use comma-separated separate boxes — NOT "Answer N:" labels between them.
_MATH_MULTI_BLANK = _compose_system(
    _ROLE_SOLVE,
    [
        "For problems with multiple [ANS] placeholders, put each sub-answer in its own "
        "\\boxed{}, separated by commas, in the order the blanks appear "
        "(e.g. \\boxed{3}, \\boxed{7}). Do not use labels like 'Answer 1:' between boxes.",
        "For single-answer problems, use one \\boxed{}.",
    ],
)

_MCQ_MULTI_BLANK = _MCQ_BASELINE

# ── Verification-forcing prompts (dev-013) ───────────────────────────────────
# Same multi-blank \\boxed{} layout as §1.3\\; adds a light verify-before-box nudge.
_VERIFY_BEFORE_BOX = (
    "Verify your answer before boxing it: substitute back, check units/range/signs, "
    "or confirm with an independent method when possible."
)

_MATH_VERIFY_PROMPT = _compose_system(
    _ROLE_SOLVE,
    [
        _VERIFY_BEFORE_BOX,
        "For problems with multiple [ANS] placeholders, put each sub-answer in its own "
        "\\boxed{}, separated by commas, in the order the blanks appear "
        "(e.g. \\boxed{3}, \\boxed{7}). Do not use labels like 'Answer 1:' between boxes.",
        "For single-answer problems, use one \\boxed{}.",
    ],
)

_MCQ_VERIFY_PROMPT = _compose_system(
    _ROLE_MCQ,
    [
        "Verify your choice before boxing it: substitute back, eliminate inconsistent options, "
        "or check constraints when possible.",
        "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}.",
    ],
)

# Variants that append the §1.3 per-item multi-blank user hint when n_blanks > 1.
_MULTI_BLANK_USER_HINT = frozenset({"multi_blank", "verify_prompt", "precision", "precision_v2", "precision_v3"})

# ── Precision / exact-form prompts (grader is 1e-8 rel-tol on 6dp gold) ───────
# Analysis (scripts/measure_precision_strict.py): 67/751 public FF fail ONLY
# because the model rounds (e.g. 62.78 vs gold 62.777778). The grader accepts
# exact forms via its symbolic path regardless of decimal places, so prefer
# fractions/radicals\\; if a decimal is unavoidable, >=8 sig figs and never round.
# Built on the multi_blank layout (keeps judger's contiguous-\boxed group rule).
_PRECISION_CLAUSE = (
    "Report each answer in EXACT form whenever possible — fractions, radicals, or "
    "symbolic constants (e.g. \\boxed{565/9} or \\boxed{3\\sqrt{10}/10}), NOT a rounded "
    "decimal. Use a decimal only when no exact form exists, and then give at least "
    "8 significant figures and do not round."
)

# Box-hygiene rules grounded in judger.is_equal probes:
#   "5 cm" != "5" (bare units fail)\\; "-1,-3" != "(-1,-3)" (brackets required)\\;
#   "50" != "0.5" (% is stripped, can't mean /100)\\; Yes/No/True/False interchange.
_GRADER_FORMAT_CLAUSE = (
    "Box the value only — no units or words (write \\boxed{5}, not \\boxed{5 cm}). "
    "Do not use a percent sign: give a probability as a decimal (e.g. 0.5) and a "
    "percentage as a bare number (e.g. 50). Keep a coordinate, tuple, or interval "
    "together in ONE box with its brackets, e.g. \\boxed{(-1,-3)} — never split it "
    "across boxes. For yes/no answers box just \\boxed{Yes} or \\boxed{No}."
)

_MATH_PRECISION = _compose_system(
    _ROLE_SOLVE,
    [
        _PRECISION_CLAUSE,
        _GRADER_FORMAT_CLAUSE,
        "For problems with multiple [ANS] placeholders, put each sub-answer in its own "
        "\\boxed{}, separated by commas, in the order the blanks appear "
        "(e.g. \\boxed{3}, \\boxed{7}). Do not use labels like 'Answer 1:' between boxes, "
        "and do not restate the boxed answers elsewhere.",
        "For single-answer problems, use one \\boxed{}.",
    ],
)

_MCQ_PRECISION = _MCQ_BASELINE

# ── Precision V2 prompts (dev-014 error analysis: 76 wrong items bucketed) ───
# Adds 6 targeted fixes to `precision` on top of multi_blank layout:
#   - Single Final Answer line, no boxed bullets       (id=44, 250\\; +4-6)
#   - Plain ", " separator (no \quad/\qquad/etc)       (id=547\\; +1-3)
#   - Multi-select MCQ as \boxed{AB} not \boxed{A,B}    (id=545\\; +1-2)
#   - Integers as 100, not 100.000                     (id=20\\; +1-3)
#   - Domain-conditional decimal-vs-exact              (id=44, 482, 495, 509, 754\\; +3-5)
#   - Slot-order check + letter-only for "which of"    (id=80, 250, 358, 391\\; +2-4)
#   - Budget hint for over-thinking truncation         (id=148, 762, ...\\; +3-5)
_FINAL_BLOCK_CLAUSE = (
    "Put your \\boxed{} answers ONLY in a single Final Answer line at the very end. "
    "Do not put \\boxed{} in bullets, headers, sub-conclusions, or 'Answer:' lines "
    "above that final line, and do not restate the boxed answers elsewhere."
)
_SEPARATOR_CLAUSE = (
    "Between final boxes use exactly a comma and a space ', '. Never put \\quad, "
    "\\qquad, \\;, \\,, \\hspace, &, or any LaTeX spacing macro between boxes "
    "(the grader's contiguous-group rule will drop boxes after such a separator)."
)
_MULTI_SELECT_CLAUSE = (
    "For a slot that asks 'select all that apply' or expects multiple option letters, "
    "concatenate the letters in alphabetical order inside ONE box with no separator: "
    "\\boxed{AB} — never \\boxed{A,B}, and never two boxes."
)
_INTEGER_FORMAT_CLAUSE = (
    "Integer answers must be plain integers with no decimal point or trailing zeros: "
    "write \\boxed{100}, not \\boxed{100.000}."
)
_PRECISION_V2_CLAUSE = (
    "For pure algebra, geometry, trig identities, and combinatorics, prefer EXACT form "
    "(fractions, radicals, \\pi). For statistics, probability, hypothesis tests, "
    "confidence intervals, finance, physics, numerical methods, and any question that "
    "says 'estimate' or 'approximate', use DECIMAL form with at least 10 significant "
    "figures of intermediate precision, and report at least 10 significant figures in "
    "the box without rounding. When in doubt, prefer decimal — the grader's numeric "
    "path is more reliable than its symbolic path."
)
_SLOT_ORDER_CLAUSE = (
    "Each \\boxed{} corresponds to ONE [ANS] in the question, in the exact order the "
    "blanks appear. Re-read the question before writing the Final Answer line and "
    "confirm slot order."
)
_LETTER_ONLY_CLAUSE = (
    "If a slot follows 'which of the following', 'select the option', or 'the correct "
    "choice is ___', return only the option LETTER (\\boxed{B}) — never the option's "
    "content (\\boxed{Yes}, \\boxed{3360})."
)
_BUDGET_HINT_CLAUSE = (
    "If your response is approaching 12,000 output characters and you have a defensible "
    "answer, stop verifying and write the Final Answer line immediately. A defensible "
    "boxed answer always beats an unfinished response."
)

_MATH_PRECISION_V2 = _compose_system(
    _ROLE_SOLVE,
    [
        _PRECISION_V2_CLAUSE,
        _INTEGER_FORMAT_CLAUSE,
        _GRADER_FORMAT_CLAUSE,
        _MULTI_SELECT_CLAUSE,
        "For problems with multiple [ANS] placeholders, put each sub-answer in its own "
        "\\boxed{}, in the order the blanks appear (e.g. \\boxed{3}, \\boxed{7}). "
        "Do not use labels like 'Answer 1:' between boxes.",
        _SLOT_ORDER_CLAUSE,
        _LETTER_ONLY_CLAUSE,
        _FINAL_BLOCK_CLAUSE,
        _SEPARATOR_CLAUSE,
        _BUDGET_HINT_CLAUSE,
    ],
)

_MCQ_PRECISION_V2 = _MCQ_BASELINE

# ── Precision V3 prompts (precision_v2 retrospective: kept what worked) ──────
# dev-014 v2 result: −2 items net (sampling noise on MCQ\\; FF flat). Per-clause
# audit on data/dev_results_precision_v2_16k.jsonl showed only the domain-
# conditional decimal clause demonstrably fired (id=495 fixed). Format clauses
# weren't validated (the items they targeted hit other failure modes this run),
# but they cost nothing. Dropped: integer-pad, slot-order, letter-only,
# budget-hint — model ignored all four.
_MATH_PRECISION_V3 = _compose_system(
    _ROLE_SOLVE,
    [
        _PRECISION_V2_CLAUSE,
        _GRADER_FORMAT_CLAUSE,
        _MULTI_SELECT_CLAUSE,
        "For problems with multiple [ANS] placeholders, put each sub-answer in its own "
        "\\boxed{}, separated by commas, in the order the blanks appear "
        "(e.g. \\boxed{3}, \\boxed{7}). Do not use labels like 'Answer 1:' between boxes.",
        _FINAL_BLOCK_CLAUSE,
        _SEPARATOR_CLAUSE,
        "For single-answer problems, use one \\boxed{}.",
    ],
)

_MCQ_PRECISION_V3 = _MCQ_BASELINE

# ── Precision XML prompts (structural tagging instead of prose) ──────────────
# Hypothesis: Qwen3-Thinking attends to structural tags more reliably than to
# embedded prose rules. Two changes vs precision_v3:
#   (1) All output rules live in a single <response_format> block at the END
#       of the system message (recency dominates inside system prompts). No
#       separate <constraints> — every rule we have describes output, not the
#       solving process, so two tags would dilute focus.
#   (2) User prompt becomes <problem><statement>…</statement>
#       [<blanks count="N"/> | <options><option letter=…>…</option>…</options>]
#       <output_format>…</output_format></problem>
# Independent of USE_XML_WRAPPERS — variant always emits this structure so the
# A/B vs precision_v3 isolates the tagging effect.

def _xml_rule_block(tag: str, rules: list[str]) -> str:
    inner = "\n".join(f"<rule>{r}</rule>" for r in rules)
    return f"<{tag}>\n{inner}\n</{tag}>"

# All rules folded into one <response_format> block. The original split into
# <constraints> + <response_format> was cosmetic — every rule below describes
# the output, not the solving process — so two tags would dilute attention
# rather than focus it. Order: value rules first (what goes in the box),
# then layout rules (where boxes go).
_PRECISION_XML_RESPONSE_RULES = [
    _PRECISION_V2_CLAUSE,
    _GRADER_FORMAT_CLAUSE,
    _MULTI_SELECT_CLAUSE,
    "Put your \\boxed{} answers ONLY in a single Final Answer line at the very "
    "end. Do not put \\boxed{} in bullets, headers, sub-conclusions, or "
    "'Answer:' lines above that final line, and do not restate the boxed "
    "answers elsewhere.",
    "Emit each final \\boxed{} EXACTLY ONCE — do not repeat the same boxed "
    "value for emphasis (e.g. never write \\boxed{G} \\boxed{G}). The "
    "grader's contiguous-group rule joins adjacent duplicate boxes into 'G, G' "
    "and rejects the answer.",
    "Between final boxes use exactly a comma and a space ', '. Never put "
    "\\quad, \\qquad, \\;, \\,, \\hspace, &, or any LaTeX spacing macro "
    "between boxes (the grader's contiguous-group rule drops boxes after such "
    "a separator).",
    "For problems with multiple [ANS] placeholders, emit one \\boxed{} per "
    "blank in the order the blanks appear (e.g. \\boxed{3}, \\boxed{7}). "
    "Do not use labels like 'Answer 1:' between boxes.",
    "For single-answer problems, use exactly one \\boxed{}.",
]

_MATH_PRECISION_XML = (
    f"<instructions>{_ROLE_SOLVE}</instructions>\n\n"
    f"{_xml_rule_block('response_format', _PRECISION_XML_RESPONSE_RULES)}"
)

_MCQ_PRECISION_XML_RESPONSE_RULES = [
    "End with a single Final Answer line containing exactly one \\boxed{} "
    "that wraps ONLY the letter of your chosen option (e.g. \\boxed{C}). "
    "Emit this box EXACTLY ONCE — never repeat it for emphasis "
    "(e.g. never write \\boxed{C} \\boxed{C}).",
    "If the slot expects multiple letters ('select all that apply'), "
    "concatenate them alphabetically inside one box with no separator "
    "(e.g. \\boxed{AB}).",
]

_MCQ_PRECISION_XML = (
    f"<instructions>{_ROLE_MCQ}</instructions>\n\n"
    f"{_xml_rule_block('response_format', _MCQ_PRECISION_XML_RESPONSE_RULES)}"
)


def _build_precision_xml_user(question: str, options) -> str:
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opt_tags = "\n".join(
            f'  <option letter="{lbl}">{opt.strip()}</option>'
            for lbl, opt in zip(labels, options)
        )
        return (
            "<problem>\n"
            f"  <statement>{question}</statement>\n"
            f"  <options>\n{opt_tags}\n  </options>\n"
            "  <output_format>End with a single Final Answer line: one "
            "\\boxed{} containing only the chosen option letter.</output_format>\n"
            "</problem>"
        )
    n_blanks = n_ans_placeholders(question)
    if n_blanks > 1:
        example = ", ".join("\\boxed{...}" for _ in range(n_blanks))
        out_fmt = (
            f"End with a single Final Answer line containing {n_blanks} "
            f"comma-separated \\boxed{{}} values in the order the [ANS] "
            f"blanks appear: {example}."
        )
        blanks_tag = f'  <blanks count="{n_blanks}"/>\n'
    else:
        out_fmt = "End with a single Final Answer line: one \\boxed{} value."
        blanks_tag = ""
    return (
        "<problem>\n"
        f"  <statement>{question}</statement>\n"
        f"{blanks_tag}"
        f"  <output_format>{out_fmt}</output_format>\n"
        "</problem>"
    )


# ── Precision XML Verify (rigor + verify + concise, all in instructions) ─────
# Two changes vs precision_xml:
#   (1) Instructions become procedural: "precise mathematical reasoner. Solve
#       rigorously, keep reasoning concise, then briefly check the final answer
#       before boxing" — covers role + verify + conciseness in one statement.
#   (2) NO separate <rule> for verify. Earlier v1 put verify in <response_format>
#       as a <rule> tag; 24% of responses echoed the tag back into output and
#       37 of those re-solved from scratch (token waste). Procedural directives
#       go in <instructions>; <response_format> stays declarative.
# Multi-blank format (separate \boxed{} per blank), dedup rule, and all other
# precision_xml rules carried over unchanged — keeps the grader-safe layout.

_ROLE_SOLVE_RIGOROUS = (
    "You are a precise mathematical reasoner. Solve the problem rigorously "
    "and keep your reasoning concise — do not re-derive steps you have already "
    "completed or go in circles. After your reasoning, briefly check your "
    "final answer for errors or contradictions (substitute back, verify "
    "units/range/signs, or confirm with an alternative method) before "
    "writing it inside \\boxed{}."
)
_ROLE_MCQ_RIGOROUS = (
    "You are a precise mathematical reasoner. Read the problem and the answer "
    "choices below, then rigorously identify the single best answer. Keep your "
    "reasoning concise. Before committing, briefly verify by elimination or "
    "by substitution into the constraints."
)

_VERIFY_RULE = (
    "Before writing the Final Answer line, independently check your work for "
    "errors or contradictions: substitute back, verify units/range/signs, or "
    "confirm with an alternative method. If you find an error, correct it and "
    "box only the corrected value."
)

_MATH_PRECISION_XML_VERIFY_RULES = [
    _PRECISION_V2_CLAUSE,
    _GRADER_FORMAT_CLAUSE,
    _MULTI_SELECT_CLAUSE,
    "Put your \\boxed{} answers ONLY in a single Final Answer line at the very "
    "end. Do not put \\boxed{} in bullets, headers, sub-conclusions, or "
    "'Answer:' lines above that final line, and do not restate the boxed "
    "answers elsewhere.",
    "Emit each final \\boxed{} EXACTLY ONCE — do not repeat the same boxed "
    "value for emphasis (e.g. never write \\boxed{G} \\boxed{G}). The "
    "grader's contiguous-group rule joins adjacent duplicate boxes into 'G, G' "
    "and rejects the answer.",
    "Between final boxes use exactly a comma and a space ', '. Never put "
    "\\quad, \\qquad, \\;, \\,, \\hspace, &, or any LaTeX spacing macro "
    "between boxes (the grader's contiguous-group rule drops boxes after such "
    "a separator).",
    "For problems with multiple [ANS] placeholders, emit one \\boxed{} per "
    "blank in the order the blanks appear (e.g. \\boxed{3}, \\boxed{7}). "
    "Do not use labels like 'Answer 1:' between boxes.",
    "For single-answer problems, use exactly one \\boxed{}.",
]

_MATH_PRECISION_XML_VERIFY = (
    f"<instructions>{_ROLE_SOLVE_RIGOROUS}</instructions>\n\n"
    f"{_xml_rule_block('response_format', _MATH_PRECISION_XML_VERIFY_RULES)}"
)

_MCQ_PRECISION_XML_VERIFY_RULES = [
    "End with a single Final Answer line containing exactly one \\boxed{} "
    "that wraps ONLY the letter of your chosen option (e.g. \\boxed{C}). "
    "Emit this box EXACTLY ONCE — never repeat it for emphasis "
    "(e.g. never write \\boxed{C} \\boxed{C}).",
    "If the slot expects multiple letters ('select all that apply'), "
    "concatenate them alphabetically inside one box with no separator "
    "(e.g. \\boxed{AB}).",
]

_MCQ_PRECISION_XML_VERIFY = (
    f"<instructions>{_ROLE_MCQ_RIGOROUS}</instructions>\n\n"
    f"{_xml_rule_block('response_format', _MCQ_PRECISION_XML_VERIFY_RULES)}"
)


# ── Select active prompts based on PROMPT_VARIANT ─────────────────────────────
_PROMPTS = {
    "baseline":       (_MATH_BASELINE, _MCQ_BASELINE),
    "concise":        (_MATH_CONCISE,  _MCQ_CONCISE),
    "multi_blank":    (_MATH_MULTI_BLANK, _MCQ_MULTI_BLANK),
    "verify_prompt": (_MATH_VERIFY_PROMPT, _MCQ_VERIFY_PROMPT),
    "precision":      (_MATH_PRECISION, _MCQ_PRECISION),
    "precision_v2":   (_MATH_PRECISION_V2, _MCQ_PRECISION_V2),
    "precision_v3":   (_MATH_PRECISION_V3, _MCQ_PRECISION_V3),
    "precision_xml":  (_MATH_PRECISION_XML, _MCQ_PRECISION_XML),
    "precision_xml_verify": (_MATH_PRECISION_XML_VERIFY, _MCQ_PRECISION_XML_VERIFY),
}
assert PROMPT_VARIANT in _PROMPTS, f"Unknown PROMPT_VARIANT {PROMPT_VARIANT!r}\\; choose from {list(_PROMPTS)}"
SYSTEM_PROMPT_MATH, SYSTEM_PROMPT_MCQ = _PROMPTS[PROMPT_VARIANT]


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if PROMPT_VARIANT in ("precision_xml", "precision_xml_verify"):
        system = SYSTEM_PROMPT_MCQ if options else SYSTEM_PROMPT_MATH
        return system, _build_precision_xml_user(question, options)
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        system, user = SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    else:
        n_blanks = n_ans_placeholders(question)
        user = question
        if PROMPT_VARIANT in _MULTI_BLANK_USER_HINT and n_blanks > 1:
            example = ", ".join("\\boxed{...}" for _ in range(n_blanks))
            user = (
                f"{question}\n\n"
                f"The problem has {n_blanks} [ANS] blanks. "
                f"After your reasoning, give {n_blanks} comma-separated \\boxed{{}} values "
                f"in order: {example}"
            )
        system, user = SYSTEM_PROMPT_MATH, user
    if USE_XML_WRAPPERS:
        user = f"<problem>{user}</problem>"
    return system, user


print(f"Active variant: {PROMPT_VARIANT!r}  USE_XML_WRAPPERS={USE_XML_WRAPPERS}")
print(f"\nMCQ system prompt:\n  {SYSTEM_PROMPT_MCQ}\n")
print(f"Math system prompt:\n  {SYSTEM_PROMPT_MATH}\n")

_demo_items = []
if mcq_sample:
    _demo_items.append(("MCQ", mcq_sample))
if free_sample:
    _demo_items.append(("Free-form", free_sample))
if multi_sample and multi_sample not in (mcq_sample, free_sample):
    _demo_items.append((f"Multi-blank ({n_ans_placeholders(multi_sample['question'])} blanks)", multi_sample))
for label, item in _demo_items:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 300 chars) ──")
    print(usr_p, "...\n")

## 7. Load model with vLLM (A100 profile)

**Not** the starter L4 INT8 path. This cell uses **bf16**, `max_model_len=32768`, prefix caching, and chunked prefill for 16k generation runs.

| Parameter | Value | Why |
|-----------|-------|-----|
| `dtype` | `bfloat16` | ~8 GB weights on A100; faster than INT8 dequant |
| `max_model_len` | 32768 | Room for `MAX_TOKENS=16384` + long thinking traces |
| `enable_prefix_caching` | True | Reuse shared system-prompt prefix across dev items |
| `enable_chunked_prefill` | True | Long prefills without OOM |
| `max_num_batched_tokens` | 32768 | Match `max_model_len` for scheduler throughput |

Full tables, observed VRAM, and L4 fallback: [`docs/infra/vllm-inference-config.md`](../docs/infra/vllm-inference-config.md). Colab install / env: [`vllm-colab-l4.md`](../docs/infra/vllm-colab-l4.md).

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# See docs/infra/vllm-inference-config.md (A100 profile)
with _jupyter_stdout_for_vllm():
    llm = LLM(
        model=MODEL_ID,
        dtype="bfloat16",
        enable_prefix_caching=True,
        gpu_memory_utilization=0.92,
        max_model_len=17500,
        trust_remote_code=True,
        max_num_seqs=256,
        max_num_batched_tokens=32768,
        enable_chunked_prefill=True,
        kv_cache_dtype="fp8"
    )

default_sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)


print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.8k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

INFO 05-31 21:36:35 [core.py:109] Initializing a V1 LLM engine (v0.20.0) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Thinking-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=17500, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=fp8, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, co

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

INFO 05-31 21:36:58 [weight_utils.py:615] Time spent downloading weights for Qwen/Qwen3-4B-Thinking-2507: 20.125687 seconds
INFO 05-31 21:36:58 [weight_utils.py:904] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 7.49 GiB. Available RAM: 79.87 GiB.
INFO 05-31 21:36:58 [weight_utils.py:927] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 05-31 21:37:00 [default_loader.py:384] Loading weights took 2.33 seconds
INFO 05-31 21:37:01 [gpu_model_runner.py:4879] Model loading took 7.61 GiB memory and 24.119065 seconds
INFO 05-31 21:37:12 [backends.py:1069] Using cache directory: /root/.cache/vllm/torch_compile_cache/d5f944a7c1/rank_0_0/backbone for vLLM's torch.compile
INFO 05-31 21:37:12 [backends.py:1128] Dynamo bytecode transform time: 10.47 s
INFO 05-31 21:37:22 [backends.py:376] Cache the graph of compile range (1, 32768) for later use
INFO 05-31 21:37:29 [backends.py:391] Compiling a graph for compile range (1, 32768) takes 15.54 s
INFO 05-31 21:37:35 [decorators.py:668] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/d274bfa70d8d3efd0218761bc4a7f48ef22308cc6339102c69338a4e0db39b80/rank_0_0/model
INFO 05-31 21:37:35 [monitor.py:53] torch.compile took 33.11 s in total
INFO 05-31 21:37:35 [monitor.py:81] Initial profiling/warmup run took 0.19 s
INFO 05-31 21:37:46 [gpu_model_run

## 7b. Budget forcing (s1-style)

When `BUDGET_FORCING=True`, §8 calls `budget_force_generate_batch` — **wave-batched** multi-pass decode (not per-item serial):

1. **Think wave:** batch `llm.generate` with `stop=[</think>]`.
2. Per output: if short think + forces left → append `Wait` and stay active; else append close tag → **finalize wave**.
3. **Finalize wave:** batch answer completion after `</think>`.

MCQ uses `BUDGET_FORCE_MIN_THINK_CHARS_MCQ` (default: no injection). FF uses `BUDGET_FORCE_MIN_THINK_CHARS`. Caps: `BUDGET_FORCE_MAX_ITER`, `BUDGET_FORCE_CONTEXT_CAP`. See [survey §A](../docs/research/2026-05-24-improvement-techniques-survey.md).

In [ ]:
THINKING_OPEN = "<think>"
THINKING_CLOSE = "</think>"


def _count_tokens(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))


def _thinking_body_chars(text: str) -> int:
    start = text.find(THINKING_OPEN)
    if start == -1:
        return len(text)
    start += len(THINKING_OPEN)
    close = text.find(THINKING_CLOSE, start)
    body = text[start:close] if close != -1 else text[start:]
    return len(body.strip())


def _clone_sampling(base: SamplingParams, **overrides) -> SamplingParams:
    fields = dict(
        max_tokens=base.max_tokens,
        temperature=base.temperature,
        top_p=base.top_p,
        top_k=base.top_k,
        min_p=base.min_p,
        presence_penalty=getattr(base, "presence_penalty", 0.0),
        repetition_penalty=getattr(base, "repetition_penalty", 1.0),
        seed=getattr(base, "seed", None),
        stop=base.stop,
    )
    fields.update(overrides)
    return SamplingParams(**fields)


def _bf_remaining_tokens(w: dict, *, context_cap_tokens: int = BUDGET_FORCE_CONTEXT_CAP) -> int:
    ctx = _count_tokens(w["prompt"] + w["generated"])
    if ctx >= context_cap_tokens:
        return 0
    return min(
        w["base_params"].max_tokens - w["total_new"],
        context_cap_tokens - ctx,
    )


def _bf_vllm_batch(prompts: list[str], params: list[SamplingParams]):
    with _jupyter_stdout_for_vllm():
        return llm.generate(prompts, sampling_params=params)


def budget_force_generate_batch(
    work: list[dict],
    *,
    max_force_iter: int = BUDGET_FORCE_MAX_ITER,
    wait_suffix: str = BUDGET_FORCE_WAIT,
    context_cap_tokens: int = BUDGET_FORCE_CONTEXT_CAP,
) -> None:
    """Wave-batched s1-style BF; mutates each work item (generated, force_count, done)."""
    active = [w for w in work if not w["done"]]
    max_waves = max_force_iter + 4

    for _ in range(max_waves):
        if not active:
            break

        think_prompts: list[str] = []
        think_sps: list[SamplingParams] = []
        think_wave: list[dict] = []

        for w in active:
            rem = _bf_remaining_tokens(w, context_cap_tokens=context_cap_tokens)
            if rem <= 0:
                w["done"] = True
                continue
            think_wave.append(w)
            think_prompts.append(w["prompt"] + w["generated"])
            think_sps.append(_clone_sampling(w["base_params"], max_tokens=rem, stop=[THINKING_CLOSE]))

        next_active: list[dict] = []
        finalize: list[dict] = []

        if think_wave:
            for w, out in zip(think_wave, _bf_vllm_batch(think_prompts, think_sps)):
                chunk = out.outputs[0].text
                w["total_new"] += len(out.outputs[0].token_ids)

                if out.outputs[0].finish_reason == "stop":
                    short = _thinking_body_chars(w["generated"] + chunk) < w["min_think_chars"]
                    if w["force_count"] < max_force_iter and short:
                        w["force_count"] += 1
                        w["generated"] = w["generated"] + chunk + wait_suffix
                        next_active.append(w)
                    else:
                        w["generated"] = w["generated"] + chunk + THINKING_CLOSE
                        finalize.append(w)
                else:
                    w["generated"] += chunk
                    w["done"] = True

        if finalize:
            fin_prompts: list[str] = []
            fin_sps: list[SamplingParams] = []
            fin_wave: list[dict] = []
            for w in finalize:
                rem = _bf_remaining_tokens(w, context_cap_tokens=context_cap_tokens)
                if rem <= 0:
                    w["done"] = True
                    continue
                fin_wave.append(w)
                fin_prompts.append(w["prompt"] + w["generated"])
                fin_sps.append(_clone_sampling(w["base_params"], max_tokens=rem, stop=None))

            if fin_wave:
                for w, out in zip(fin_wave, _bf_vllm_batch(fin_prompts, fin_sps)):
                    w["generated"] += out.outputs[0].text
                    w["total_new"] += len(out.outputs[0].token_ids)
                    w["done"] = True

        active = [w for w in next_active if not w["done"]]

    for w in work:
        w["generated"] = w["generated"].strip()


print("Budget forcing helpers loaded." if BUDGET_FORCING else "Budget forcing disabled (helpers available).")

## 7c. Progressive-hint prompting (PHP)

When `PHP_ENABLED=True`, §8b runs a second pass on FF items only:

1. **Pass 1:** results of §8 generation (with or without budget forcing).
2. Extract tentative `\boxed{...}` from pass 1 via `extract_boxed_raw` (same brace-matched logic as `judger.extract_all_boxed`, no normalization).
3. **Pass 2:** re-issue the prompt with a neutral *"a previous attempt got X — independently re-solve and confirm or revise"* instruction appended to the user message. The original system+user prefix is preserved so vLLM prefix-cache hits.
4. Final response = pass-2 text. Pass-1 text saved to `<output>.php_pass1.jsonl` for analysis (flip rate, anchor failures, etc.).

MCQ skipped by default (`PHP_FF_ONLY=True`). If pass-1 has no `\boxed{}` and `PHP_SKIP_IF_NO_PASS1_ANSWER=True`, item keeps its pass-1 response unchanged. See [survey §1.13](../docs/research/2026-05-24-improvement-techniques-survey.md).

In [ ]:
# {format_clause} is filled per-item in build_php_prompt so multi-blank FF gets the
# same \boxed{a}, \boxed{b} instruction the §1.3 prompt uses. Without it, pass-2
# drifts into labeled "(a) \boxed{...} (b) ... \boxed{...}" layout that fails
# judger.extract_all_boxed's contiguous-group rule (see dev-011-php R→W breaks).
PHP_HINT_TEMPLATE = (
    "A previous attempt at this problem produced the answer: {prev_answer}\n\n"
    "Independently re-solve the problem step-by-step. "
    "If your fresh derivation matches that answer, confirm it; "
    "if it differs, give your corrected answer. "
    "{format_clause}"
)

PHP_FORMAT_DEFAULT = "Put your final answer inside \\boxed{} as before."


def _php_format_clause(item: dict) -> str:
    """Return the closing format sentence for the PHP hint.

    For multi-blank FF items, enforces the §1.3 multi_blank layout so pass-2 does
    not produce prose-separated `\\boxed{}` that judger's contiguous-group rule
    cannot join. MCQ and single-blank fall back to the default single-box clause.
    """
    if item.get("options"):
        return PHP_FORMAT_DEFAULT
    n_blanks = n_ans_placeholders(item["question"])
    if n_blanks > 1:
        example = ", ".join("\\boxed{...}" for _ in range(n_blanks))
        return (
            f"The problem has {n_blanks} [ANS] blanks. Give {n_blanks} comma-separated "
            f"\\boxed{{}} values in the order the blanks appear: {example}. "
            f"Do not use labels like '(a)', 'Answer 1:', or sentences between boxes — "
            f"the boxes must be adjacent so the grader can group them."
        )
    return PHP_FORMAT_DEFAULT


def extract_boxed_raw(text: str) -> str | None:
    """Last contiguous group of raw \\boxed{...} contents from `text` (post-</think>).

    Mirrors judger.extract_all_boxed brace-depth logic but skips normalization so the
    hint passed to pass-2 reads as the model originally wrote it. Returns None when no
    valid box is found.
    """
    think_end = text.rfind(THINKING_CLOSE)
    search = text[think_end + len(THINKING_CLOSE):] if think_end >= 0 else text

    entries: list[tuple[int, int, str]] = []
    start = 0
    while True:
        idx = search.find("\\boxed{", start)
        if idx < 0:
            break
        brace_start = idx + len("\\boxed{")
        depth = 1
        i = brace_start
        while i < len(search) and depth > 0:
            ch = search[i]
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
            i += 1
        if depth == 0:
            content = search[brace_start:i - 1].strip()
            if content:
                entries.append((idx, i, content))
        start = i

    if not entries:
        return None

    last_group = [entries[-1]]
    for j in range(len(entries) - 2, -1, -1):
        gap = search[entries[j][1]:entries[j + 1][0]]
        if re.match(r"^[\s,\$\.\;\:\-\&\\]*$", gap):
            last_group.insert(0, entries[j])
        else:
            break
    return ", ".join(e[2] for e in last_group)


def build_php_prompt(item: dict, prev_answer: str) -> str:
    """Pass-2 chat-template string. Same system/user as pass-1 + hint appended to user."""
    system, user = build_prompt(item["question"], item.get("options"))
    hint = PHP_HINT_TEMPLATE.format(
        prev_answer=prev_answer,
        format_clause=_php_format_clause(item),
    )
    user_with_hint = f"{user}\n\n---\n{hint}"
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user_with_hint}],
        tokenize=False,
        add_generation_prompt=True,
    )


print("PHP helpers loaded." if PHP_ENABLED else "PHP disabled (helpers available).")


## 7d. Self-consistency helpers (extract → canon → vote → tiebreak)

Loaded before §8 when `SELF_CONSISTENCY=True`. §8 generates `SamplingParams(n=SC_K)` traces and merges inline; §8c can re-merge from `<output>.sc_traces.jsonl` after changing anchor. Incompatible with `BUDGET_FORCING`.


In [ ]:
_judger_override = os.environ.get("CSE151B_JUDGER_DIR", "").strip()
try:
    _drive_root = str(DRIVE_PROJECT_ROOT)
except NameError:
    _drive_root = ""
JUDGER_ROOT = _judger_override or _drive_root or str(REPO_ROOT)

print(f"JUDGER_ROOT={JUDGER_ROOT}")
sys.path.insert(0, os.path.abspath(JUDGER_ROOT))
from judger import Judger
judger = Judger(strict_extract=False)

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

import sympy
from collections import Counter
from sympy.parsing.latex import parse_latex

CANON_NUMERIC_PREC = 6
_SET_INTERVAL_RE = re.compile(r"^[\{\[\(].*[\}\]\)]$")


def _strip_units_for_canon(s: str) -> str:
    s = s.strip()
    s = re.sub(r"\\,.*$", "", s).strip()
    s = re.sub(r"\s*\\text\{[^}]*\}\s*$", "", s).strip()
    return s


def _split_multi_value_box(s: str) -> list[str]:
    s = s.strip()
    if not s or _SET_INTERVAL_RE.match(s):
        return [s]
    if "," not in s:
        return [s]
    parts, depth, start = [], 0, 0
    for i, ch in enumerate(s):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth = max(0, depth - 1)
        elif ch == "," and depth == 0:
            parts.append(s[start:i].strip())
            start = i + 1
    parts.append(s[start:].strip())
    return [p for p in parts if p] if len(parts) > 1 else [s]


def _canon(s: str) -> str:
    s = _strip_units_for_canon(s)
    if not s:
        return ""
    if _SET_INTERVAL_RE.match(s):
        return s.replace(" ", "").lower()
    for parser in (parse_latex, sympy.sympify):
        try:
            expr = parser(s)
            if expr.is_number or getattr(expr, "is_Number", False):
                f = float(expr.evalf())
                return format(round(f, CANON_NUMERIC_PREC), f".{CANON_NUMERIC_PREC}g")
            return str(sympy.simplify(expr))
        except Exception:
            pass
    s = s.replace(" ", "")
    s = s.replace("\\frac", "/").replace("\\sqrt", "sqrt")
    s = s.replace("{", "").replace("}", "")
    return s.lower()


def _canon_box_list(boxes: list[str]) -> list[str]:
    out: list[str] = []
    for b in boxes:
        for part in _split_multi_value_box(b):
            out.append(_canon(part))
    return out


def _extract_boxes(text: str) -> list[str] | None:
    if THINKING_CLOSE not in text:
        return None
    try:
        boxes = judger.extract_all_boxed(text)
    except Exception:
        return None
    return boxes if boxes else None


def _prepare_ff_boxes(boxes: list[str], n_blanks: int) -> list[str] | None:
    if not boxes:
        return None
    if n_blanks <= 1:
        return [boxes[-1]]
    if len(boxes) != n_blanks:
        return None
    return boxes


def _vote_ff(
    trace_box_lists: list[list[str]],
    n_blanks: int,
    anchor: list[str],
) -> tuple[list[str], int]:
    canon_tuples = [
        tuple(_canon_box_list(t))
        for t in trace_box_lists
        if len(_canon_box_list(t)) == (1 if n_blanks <= 1 else n_blanks)
    ]
    if not canon_tuples:
        return anchor, 0

    counts = Counter(canon_tuples)
    top_tuple, top_n = counts.most_common(1)[0]
    return list(top_tuple), top_n


def _extract_mcq_letter(text: str) -> str | None:
    if THINKING_CLOSE not in text:
        return None
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    return m.group(1).upper() if m else None


def _vote_mcq(letters: list[str], anchor: str) -> str:
    valid = [L for L in letters if L]
    if not valid:
        return anchor
    counts = Counter(valid)
    top_n = counts.most_common(1)[0][1]
    tied = [k for k, v in counts.items() if v == top_n]
    if len(tied) == 1:
        return tied[0]
    if anchor in tied:
        return anchor
    return sorted(tied)[0]


def _ff_boxes_match_vote(boxes: list[str], voted: list[str], n_blanks: int) -> bool:
    prep = _prepare_ff_boxes(boxes, n_blanks)
    if prep is None:
        return False
    return tuple(_canon_box_list(prep)) == tuple(voted)


def _pick_sc_trace(
    traces: list[str],
    *,
    is_mcq: bool,
    voted,
    anchor_trace: str,
    n_blanks: int,
) -> str:
    if is_mcq:
        voted_letter = voted
        candidates = [t for t in traces if _extract_mcq_letter(t) == voted_letter]
    else:
        voted_boxes = voted
        candidates = [
            t for t in traces
            if _ff_boxes_match_vote(_extract_boxes(t) or [], voted_boxes, n_blanks)
        ]

    if not candidates:
        return anchor_trace

    if anchor_trace in candidates:
        return anchor_trace

    valid = [t for t in candidates if THINKING_CLOSE in t]
    pool = valid or candidates
    return max(pool, key=len)


def _load_sc_anchor_map() -> dict[int, str]:
    if not SC_ANCHOR_PATH:
        return {}
    path = Path(SC_ANCHOR_PATH)
    if not path.is_file():
        print(f"SC_ANCHOR_PATH not found: {path} — using sample 0 per item")
        return {}
    out: dict[int, str] = {}
    with open(path) as f:
        for line in f:
            r = json.loads(line)
            out[int(r["id"])] = r["response"]
    print(f"SC anchor loaded: {len(out)} traces from {path.name}")
    return out


def sc_merge_traces(traces: list[str], item: dict, anchor_trace: str) -> tuple[str, dict]:
    n_total = len(traces)
    is_mcq = bool(item.get("options"))
    if is_mcq:
        anchor_letter = _extract_mcq_letter(anchor_trace) or ""
        letters = []
        n_dropped = sum(1 for t in traces if THINKING_CLOSE not in t)
        for t in traces:
            let = _extract_mcq_letter(t)
            if let:
                letters.append(let)
        n_valid = len(letters)
        voted = _vote_mcq(letters, anchor_letter)
        top_count = Counter(letters)[voted] if letters else 0
        stats = {"n_traces": n_total, "n_dropped": n_dropped, "n_valid": n_valid,
                 "top_count": top_count, "agreed": top_count > n_valid / 2}
        response = _pick_sc_trace(
            traces, is_mcq=True, voted=voted, anchor_trace=anchor_trace, n_blanks=1,
        )
        return response, stats

    n_blanks = n_ans_placeholders(item["question"])
    anchor_boxes = _prepare_ff_boxes(_extract_boxes(anchor_trace) or [], n_blanks)
    anchor_canon = _canon_box_list(anchor_boxes) if anchor_boxes else []

    n_dropped = sum(1 for t in traces if THINKING_CLOSE not in t)
    trace_box_lists: list[list[str]] = []
    for t in traces:
        raw = _extract_boxes(t)
        prep = _prepare_ff_boxes(raw or [], n_blanks)
        if prep is not None:
            trace_box_lists.append(prep)

    n_valid = len(trace_box_lists)
    voted, top_count = _vote_ff(trace_box_lists, n_blanks, anchor_canon)
    stats = {"n_traces": n_total, "n_dropped": n_dropped, "n_valid": n_valid,
             "top_count": top_count, "agreed": top_count > n_valid / 2}
    response = _pick_sc_trace(
        traces,
        is_mcq=False,
        voted=voted,
        anchor_trace=anchor_trace,
        n_blanks=n_blanks,
    )
    return response, stats


def apply_sc_merge(completed: dict, sc_traces_by_id: dict[int, list[str]], anchor_map: dict[int, str]) -> int:
    n_merged = 0
    by_id = {d["id"]: d for d in data}
    for qid, traces in sc_traces_by_id.items():
        item = by_id[qid]
        anchor = anchor_map.get(qid, traces[0])
        response, _ = sc_merge_traces(traces, item, anchor)
        completed[qid] = response
        n_merged += 1
    return n_merged


print("SC helpers loaded." if SELF_CONSISTENCY else "SC helpers available (SELF_CONSISTENCY=False).")


## 8. Generate responses

Checkpoint: `results/dev_results_{variant}_{Nk}[_bf][_php][_scK][_smoke].responses.jsonl` (Drive on Colab).

- **`BUDGET_FORCING=False`:** batched `llm.generate` (starter path).
- **`BUDGET_FORCING=True`:** wave-batched multi-pass decode (§7b); ~1–3 vLLM batches per chunk wave, not per-item serial.

Delete the checkpoint file to regenerate from scratch.

When `SELF_CONSISTENCY=True` (and `BUDGET_FORCING=False`), uses `SamplingParams(n=SC_K)` so each prompt yields K traces; §7d merges inline (§8c can re-merge from checkpoint). Raw K traces → `<output>.sc_traces.jsonl`. Delete old `.responses.jsonl` when enabling SC on a prior K=1 run.


In [26]:
CHUNK_SIZE = len(data)

try:
    _results_dir = DRIVE_PROJECT_ROOT / "results"
except NameError:
    _results_dir = Path(OUTPUT_PATH).parent

_results_dir.mkdir(parents=True, exist_ok=True)
response_checkpoint = _results_dir / (Path(OUTPUT_PATH).stem + ".responses.jsonl")
sc_traces_checkpoint = _results_dir / (Path(OUTPUT_PATH).stem + ".sc_traces.jsonl")
print(f"Checkpoint path: {response_checkpoint}")

completed: dict = {}
if response_checkpoint.exists():
    with open(response_checkpoint) as f:
        for line in f:
            r = json.loads(line)
            completed[r["id"]] = r["response"]
    print(f"Resumed from checkpoint: {len(completed)} responses already generated")

remaining = [d for d in data if d.get("id") not in completed]
print(
    f"Generating {len(remaining)} remaining / {len(data)} total"
    + (" (budget forcing, wave-batched)" if BUDGET_FORCING
       else f" (batched, SC K={SC_K})" if SELF_CONSISTENCY else " (batched)")
    + "..."
)

_sc_stats_log: list[dict] = []
bf_stats: dict[int, int] = {}
bf_is_mcq: dict[int, bool] = {}

for chunk_start in range(0, len(remaining), CHUNK_SIZE):
    chunk = remaining[chunk_start : chunk_start + CHUNK_SIZE]

    if SELF_CONSISTENCY and BUDGET_FORCING:
        raise RuntimeError("SELF_CONSISTENCY and BUDGET_FORCING are mutually exclusive.")

    if BUDGET_FORCING:
        work = []
        for item in chunk:
            system, user = build_prompt(item["question"], item.get("options"))
            prompt_text = tokenizer.apply_chat_template(
                [{"role": "system", "content": system},
                 {"role": "user",   "content": user}],
                tokenize=False,
                add_generation_prompt=True,
            )
            is_mcq = bool(item.get("options"))
            work.append({
                "id": item["id"],
                "prompt": prompt_text,
                "base_params": (
                    default_sampling_params
                ),
                "generated": "",
                "force_count": 0,
                "total_new": 0,
                "min_think_chars": (
                    BUDGET_FORCE_MIN_THINK_CHARS_MCQ if is_mcq else BUDGET_FORCE_MIN_THINK_CHARS
                ),
                "is_mcq": is_mcq,
                "done": False,
            })

        budget_force_generate_batch(work)

        with open(response_checkpoint, "a") as f:
            for w in work:
                bf_stats[w["id"]] = w["force_count"]
                bf_is_mcq[w["id"]] = w["is_mcq"]
                completed[w["id"]] = w["generated"]
                f.write(json.dumps({"id": w["id"], "response": w["generated"]}) + "\n")
    else:
        prompts = []
        for item in chunk:
            system, user = build_prompt(item["question"], item.get("options"))
            prompt_text = tokenizer.apply_chat_template(
                [{"role": "system", "content": system},
                 {"role": "user",   "content": user}],
                tokenize=False,
                add_generation_prompt=True,
            )
            prompts.append(prompt_text)

        if SELF_CONSISTENCY:
            sc_params = _clone_sampling(default_sampling_params, n=SC_K)
            chunk_params = [sc_params] * len(prompts)
        else:
            chunk_params = [default_sampling_params] * len(prompts)

        with _jupyter_stdout_for_vllm():
            outputs = llm.generate(prompts, sampling_params=chunk_params)

        anchor_map = _load_sc_anchor_map() if SELF_CONSISTENCY else {}

        with open(response_checkpoint, "a") as f:
            sc_append = None
            if SELF_CONSISTENCY:
                sc_append = open(sc_traces_checkpoint, "a")
            try:
                for item, out in zip(chunk, outputs):
                    if SELF_CONSISTENCY:
                        traces = [o.text.strip() for o in out.outputs]
                        anchor = anchor_map.get(item["id"], traces[0])
                        response, sc_stats = sc_merge_traces(traces, item, anchor)
                        _sc_stats_log.append({"id": item["id"], **sc_stats})
                        sc_append.write(json.dumps({"id": item["id"], "traces": traces}) + "\n")
                    else:
                        response = out.outputs[0].text.strip()
                    completed[item["id"]] = response
                    f.write(json.dumps({"id": item["id"], "response": response}) + "\n")
            finally:
                if sc_append is not None:
                    sc_append.close()

    done = len(completed)
    print(f"  {done}/{len(data)}  ({done / len(data) * 100:.1f}%)")

responses = [completed[d["id"]] for d in data]
if BUDGET_FORCING and bf_stats:

    def _bf_rate(ids: list[int]) -> str:
        if not ids:
            return "n/a"
        n_forced = sum(1 for i in ids if bf_stats.get(i, 0) > 0)
        mean_inj = sum(bf_stats[i] for i in ids) / len(ids)
        return f"{n_forced}/{len(ids)} forced ({100 * n_forced / len(ids):.1f}%), mean {mean_inj:.2f} inj/item"

    mcq_ids = [i for i, m in bf_is_mcq.items() if m]
    ff_ids = [i for i, m in bf_is_mcq.items() if not m]
    print(f"Budget forcing MCQ: {_bf_rate(mcq_ids)}")
    print(f"Budget forcing FF : {_bf_rate(ff_ids)}")

    # ── Think-body length distribution (calibrates min_think_chars threshold) ──
    def _think_body_len(text: str) -> int:
        close = text.find(THINKING_CLOSE)
        body = text[:close] if close != -1 else text
        return len(body.strip())

    def _summarize_lens(ids: list[int], label: str) -> None:
        if not ids:
            print(f"  {label}: n=0")
            return
        lens = sorted(_think_body_len(completed[i]) for i in ids)
        n = len(lens)
        p = lambda q: lens[min(n - 1, int(q * n))]
        print(
            f"  {label}: n={n}  "
            f"p25={p(0.25):>6,}  p50={p(0.50):>6,}  p75={p(0.75):>6,}  "
            f"min={lens[0]:>6,}  max={lens[-1]:>6,}  "
            f"< {BUDGET_FORCE_MIN_THINK_CHARS}: {sum(1 for L in lens if L < BUDGET_FORCE_MIN_THINK_CHARS)}/{n}"
        )

    print("Think-body chars (final generated text up to </think>):")
    _summarize_lens(ff_ids, "FF ")
    _summarize_lens(mcq_ids, "MCQ")
if SELF_CONSISTENCY:
    print(f"SC traces checkpoint: {sc_traces_checkpoint.name}")
    if _sc_stats_log:
        n_sc = len(_sc_stats_log)
        avg_valid   = sum(s["n_valid"]   for s in _sc_stats_log) / n_sc
        avg_dropped = sum(s["n_dropped"] for s in _sc_stats_log) / n_sc
        avg_top     = sum(s["top_count"] for s in _sc_stats_log) / n_sc
        n_agreed    = sum(1 for s in _sc_stats_log if s["agreed"])
        print(f"SC stats ({n_sc} items, K={SC_K}):")
        print(f"  valid/K        : {avg_valid:.2f}  (dropped {avg_dropped:.2f}/item — no </think>)")
        print(f"  top vote count : {avg_top:.2f}  agreed(>50%)={n_agreed}/{n_sc} ({100*n_agreed/n_sc:.1f}%)")
print(f"Done. {len(responses)} responses ready.")

Rendering prompts:   0%|          | 0/225 [00:00<?, ?it/s]

Checkpoint path: /content/drive/MyDrive/CSE151B/results/dev_results_precision_xml_verify_16k.responses.jsonl
Generating 225 remaining / 225 total (batched)...


Processed prompts:   0%|          | 0/225 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  225/225  (100.0%)
Done. 225 responses ready.


In [ ]:
# §8c — re-merge SC from checkpoint (optional; §8 already merges on generate)
if SELF_CONSISTENCY:
    if BUDGET_FORCING:
        raise RuntimeError("SELF_CONSISTENCY and BUDGET_FORCING are mutually exclusive.")

    sc_traces_path = _results_dir / (Path(OUTPUT_PATH).stem + ".sc_traces.jsonl")
    sc_traces_by_id: dict[int, list[str]] = {}

    if sc_traces_path.exists():
        with open(sc_traces_path) as f:
            for line in f:
                r = json.loads(line)
                sc_traces_by_id[int(r["id"])] = r["traces"]

    missing = [d["id"] for d in data if d["id"] not in sc_traces_by_id]
    if missing:
        print(f"SC re-merge: {len(missing)} ids missing in {sc_traces_path.name} — run §8 first.")
    if sc_traces_by_id:
        anchor_map = _load_sc_anchor_map()
        n = apply_sc_merge(completed, sc_traces_by_id, anchor_map)
        responses = [completed[d["id"]] for d in data]
        print(f"SC re-merge applied: {n} items (anchor map size {len(anchor_map)})")
else:
    print("SELF_CONSISTENCY disabled — skipping §8c merge.")


## 8b. PHP pass-2 (FF only)

Runs second pass for free-form items using §8 outputs as the pass-1 trace. Pass-1 saved to `<output>.php_pass1.jsonl`; `completed[id]` overwritten with pass-2. Resumable via `<output>.php_pass2.jsonl` checkpoint.

In [ ]:
THINKING_OPEN = "<think>"
THINKING_CLOSE = "</think>"

def _clone_sampling(base: SamplingParams, **overrides) -> SamplingParams:
    fields = dict(
        max_tokens=base.max_tokens,
        temperature=base.temperature,
        top_p=base.top_p,
        top_k=base.top_k,
        min_p=base.min_p,
        presence_penalty=getattr(base, "presence_penalty", 0.0),
        repetition_penalty=getattr(base, "repetition_penalty", 1.0),
        seed=getattr(base, "seed", None),
        stop=base.stop,
    )
    fields.update(overrides)
    return SamplingParams(**fields)

if PHP_ENABLED:
    php_targets = [
        d for d in data
        if (not PHP_FF_ONLY) or (not d.get("options"))
    ]

    php_pass1_path = _results_dir / (Path(OUTPUT_PATH).stem + ".php_pass1.jsonl")
    php_pass2_path = _results_dir / (Path(OUTPUT_PATH).stem + ".php_pass2.jsonl")

    # Snapshot pass-1 for every PHP target (overwrite: pass-1 is whatever §8 just wrote).
    with open(php_pass1_path, "w") as f:
        for d in php_targets:
            f.write(json.dumps({"id": d["id"], "response": completed[d["id"]]}) + "\n")
    print(f"PHP pass-1 snapshot: {len(php_targets)} items → {php_pass1_path.name}")

    php_done: dict = {}
    if php_pass2_path.exists():
        with open(php_pass2_path) as f:
            for line in f:
                r = json.loads(line)
                php_done[r["id"]] = r["response"]
        print(f"PHP pass-2 checkpoint: {len(php_done)} already done")

    by_id = {d["id"]: d for d in php_targets}
    todo_ids = [d["id"] for d in php_targets if d["id"] not in php_done]

    skipped_no_answer: list[int] = []
    php_prompts: list[str] = []
    php_params: list[SamplingParams] = []
    php_batch_ids: list[int] = []

    for qid in todo_ids:
        item = by_id[qid]
        prev = extract_boxed_raw(completed[qid])
        if prev is None and PHP_SKIP_IF_NO_PASS1_ANSWER:
            skipped_no_answer.append(qid)
            continue
        prev_for_hint = prev if prev is not None else "(no boxed answer extracted)"
        php_prompts.append(build_php_prompt(item, prev_for_hint))
        php_params.append(_clone_sampling(default_sampling_params, max_tokens=PHP_MAX_TOKENS))
        php_batch_ids.append(qid)

    print(
        f"PHP pass-2: {len(php_batch_ids)} to generate, "
        f"{len(php_done)} cached, {len(skipped_no_answer)} skipped (no pass-1 box)"
    )

    if php_batch_ids:
        with _jupyter_stdout_for_vllm():
            outs = llm.generate(php_prompts, sampling_params=php_params)
        with open(php_pass2_path, "a") as f:
            for qid, out in zip(php_batch_ids, outs):
                resp = out.outputs[0].text.strip()
                php_done[qid] = resp
                f.write(json.dumps({"id": qid, "response": resp}) + "\n")

    # Overwrite final responses with pass-2 where available.
    n_replaced = 0
    for qid in [d["id"] for d in php_targets]:
        if qid in php_done:
            completed[qid] = php_done[qid]
            n_replaced += 1
    responses = [completed[d["id"]] for d in data]
    print(f"PHP pass-2 applied: {n_replaced} responses replaced, "
          f"{len(skipped_no_answer)} kept pass-1 unchanged")
else:
    print("PHP disabled — skipping pass-2.")


## 9. Score responses (same as starter)

In [27]:
_judger_override = os.environ.get("CSE151B_JUDGER_DIR", "").strip()
try:
    _drive_root = str(DRIVE_PROJECT_ROOT)
except NameError:
    _drive_root = ""
JUDGER_ROOT = _judger_override or _drive_root or str(REPO_ROOT)

print(f"JUDGER_ROOT={JUDGER_ROOT}")
sys.path.insert(0, os.path.abspath(JUDGER_ROOT))
from judger import Judger
judger = Judger(strict_extract=False)

JUDGER_ROOT=/content/drive/MyDrive/CSE151B


## 8d. SC canonicalization unit test (pub-002 wrong FF)

Offline check on `data/full_public_16k.jsonl`: for FF items marked wrong, compare `_canon(extracted box)` to `_canon(gold)` — surfaces canon bugs without extra sampling.

In [ ]:
_PUB002_RESULTS = REPO_ROOT / "data" / "full_public_16k.jsonl"
_PUB002_RESPONSES = REPO_ROOT / "data" / "full_public_16k.responses.jsonl"

if not (_PUB002_RESULTS.is_file() and _PUB002_RESPONSES.is_file()):
    print("pub-002 artifacts missing — skip canon unit test.")
else:
    if "judger" not in globals():
        sys.path.insert(0, str(REPO_ROOT))
        from judger import Judger
        judger = Judger(strict_extract=False)

    with open(_PUB002_RESULTS) as f:
        pub_rows = [json.loads(l) for l in f]
    with open(_PUB002_RESPONSES) as f:
        resp_by_id = {int(json.loads(l)["id"]): json.loads(l)["response"] for l in f}

    wrong_ff = [
        r for r in pub_rows
        if not r.get("is_mcq") and not r.get("correct")
    ]
    n_would_fix = 0
    n_checked = 0
    samples = []
    for row in wrong_ff[:500]:
        qid = row["id"]
        resp = resp_by_id.get(qid)
        if not resp:
            continue
        gold = row["gold"] if isinstance(row["gold"], list) else [row["gold"]]
        n_blanks = len(gold)
        boxes = _prepare_ff_boxes(_extract_boxes(resp) or [], n_blanks)
        if boxes is None:
            continue
        n_checked += 1
        pred_c = _canon_box_list(boxes)
        gold_c = [_canon(g) for g in gold]
        if pred_c == gold_c:
            n_would_fix += 1
            if len(samples) < 8:
                samples.append((qid, boxes, gold, pred_c, gold_c))

    print(f"pub-002 wrong FF canon check: {n_would_fix}/{n_checked} would match gold after canon only")
    if samples:
        print("  sample ids (extraction was right, grading/normalization failed):")
        for qid, boxes, gold, pc, gc in samples:
            print(f"    id={qid}  boxes={boxes!r}  gold={gold!r}")
            print(f"           canon_pred={pc}  canon_gold={gc}")


In [28]:
# Route MCQ through the same Judger as free-form so dev scoring matches the
# official competition extractor exactly (handles </think> stripping, last
# contiguous \boxed{} group, and multi-select MCQ like \boxed{AB}).
def score_mcq(response: str, gold_letter: str) -> bool:
    extracted = (judger.extract_ans(response) or "").strip().upper()
    gold = str(gold_letter).strip().upper()
    # Multi-select gold like "AB" → use judge_MC_multiple; single letter → judge_MC_single
    if len(gold) > 1:
        return judger.judge_MC_multiple(extracted, gold)
    return judger.judge_MC_single(extracted, gold)


results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")


Scoring: 100%|██████████| 225/225 [00:20<00:00, 10.83it/s]

Scoring complete. 225 results.


## 9b. PHP analysis (if `PHP_ENABLED`)

Re-scores the pass-1 snapshot (`<output>.php_pass1.jsonl`) and final responses to quantify what PHP's second pass actually changed: item-level R→W / W→R transitions, multi-blank vs single-blank deltas, and the specific ids that flipped. Skipped when `PHP_ENABLED=False`.

Mirrors the offline analysis in [docs/log/runs/dev-011-php.md](../docs/log/runs/dev-011-php.md). Run before §10 so the summary row can cite within-run deltas.

In [ ]:
if not PHP_ENABLED:
    print("PHP disabled — skipping analysis.")
else:
    import contextlib
    import io

    _ff_items = [d for d in data if not d.get("options")]
    _q_by_id  = {d["id"]: d["question"] for d in data}
    _gold_by  = {d["id"]: (d["answer"] if isinstance(d["answer"], list) else [d["answer"]])
                 for d in _ff_items}

    php_pass1_path = _results_dir / (Path(OUTPUT_PATH).stem + ".php_pass1.jsonl")
    php_pass2_path = _results_dir / (Path(OUTPUT_PATH).stem + ".php_pass2.jsonl")

    if not php_pass1_path.exists():
        print(f"Missing {php_pass1_path.name} — re-run §8b first.")
    else:
        with open(php_pass1_path) as f:
            pass1 = {json.loads(l)["id"]: json.loads(l)["response"] for l in f}
        pass2 = {}
        if php_pass2_path.exists():
            with open(php_pass2_path) as f:
                for l in f:
                    r = json.loads(l)
                    pass2[r["id"]] = r["response"]   # last write wins on resume

        # Final = pass2 if present, else pass1 (PHP_SKIP_IF_NO_PASS1_ANSWER path)
        final = {qid: pass2.get(qid, pass1[qid]) for qid in pass1}

        def _ff_score(resp, gold):
            with contextlib.redirect_stdout(io.StringIO()):
                try:
                    return judger.auto_judge(pred=resp, gold=gold, options=[[]] * len(gold))
                except Exception:
                    return False

        def _bucket(qid):
            return "multi" if n_ans_placeholders(_q_by_id[qid]) > 1 else "single"

        def _tally(pred_map):
            n=c=mn=mc=sn=sc=0
            ok_by_id = {}
            for qid, resp in pred_map.items():
                if qid not in _gold_by:
                    continue
                ok = _ff_score(resp, _gold_by[qid])
                ok_by_id[qid] = ok
                n += 1; c += int(ok)
                if _bucket(qid) == "multi":
                    mn += 1; mc += int(ok)
                else:
                    sn += 1; sc += int(ok)
            return c, n, mc, mn, sc, sn, ok_by_id

        p1_c, p1_n, p1_mc, p1_mn, p1_sc, p1_sn, p1_ok = _tally(pass1)
        fn_c, fn_n, fn_mc, fn_mn, fn_sc, fn_sn, fn_ok = _tally(final)

        def _pct(c, n): return f"{100*c/n:5.2f}%" if n else "  —  "
        def _delta(a, b, n): return f"{100*(b-a)/n:+5.2f} pp" if n else "  —  "

        print("=" * 62)
        print(f"PHP within-run FF analysis  ({p1_n} FF items, {len(pass2)} got pass-2)")
        print("=" * 62)
        print(f"{'Slice':<14}{'Pass-1':>14}{'Final':>14}{'Δ':>14}")
        print("-" * 56)
        print(f"{'FF overall':<14}{p1_c}/{p1_n} {_pct(p1_c,p1_n):>7}"
              f"   {fn_c}/{fn_n} {_pct(fn_c,fn_n):>7}   {_delta(p1_c,fn_c,p1_n):>10}")
        print(f"{'Multi-blank':<14}{p1_mc}/{p1_mn} {_pct(p1_mc,p1_mn):>7}"
              f"   {fn_mc}/{fn_mn} {_pct(fn_mc,fn_mn):>7}   {_delta(p1_mc,fn_mc,p1_mn):>10}")
        print(f"{'Single-blank':<14}{p1_sc}/{p1_sn} {_pct(p1_sc,p1_sn):>7}"
              f"   {fn_sc}/{fn_sn} {_pct(fn_sc,fn_sn):>7}   {_delta(p1_sc,fn_sc,p1_sn):>10}")

        rr=rw=wr=ww=0; rw_ids=[]; wr_ids=[]
        for qid in p1_ok:
            a, b = p1_ok[qid], fn_ok.get(qid, p1_ok[qid])
            if a and b: rr += 1
            elif a and not b: rw += 1; rw_ids.append(qid)
            elif (not a) and b: wr += 1; wr_ids.append(qid)
            else: ww += 1
        print("-" * 56)
        print(f"Transitions (pass-1 → final):")
        print(f"  R→R {rr:3d}   W→R {wr:3d}  ids={wr_ids}")
        print(f"  W→W {ww:3d}   R→W {rw:3d}  ids={rw_ids}")
        print(f"  Net items moved: {wr - rw:+d}")
        print("=" * 62)

        # Per-item box diff for the items that flipped.
        # Uses judger.extract_all_boxed — same contiguous-group rule that scoring uses.
        def _last_boxes(text):
            try:
                return judger.extract_all_boxed(text)
            except Exception:
                return []

        flip_ids = sorted(set(rw_ids + wr_ids))
        if flip_ids:
            print("\nPer-item box diff for flips (boxes shown post-judger.extract_all_boxed):")
            print(f"{'id':>5}  {'verdict':<7}  {'blanks':>6}  pass-1 → pass-2")
            for qid in flip_ids:
                verdict = "W→R" if qid in wr_ids else "R→W"
                p1_box = _last_boxes(pass1[qid])
                p2_box = _last_boxes(pass2.get(qid, pass1[qid]))
                nb_ = n_ans_placeholders(_q_by_id[qid])
                print(f"{qid:>5}  {verdict:<7}  {nb_:>6}  {p1_box}  →  {p2_box}")


## 10. Summary & registry

Prints dev metrics and a copy-paste row for [`docs/log/experiments.md`](../docs/log/experiments.md). Recorded runs: [dev-007](../docs/log/runs/dev-007-max-tokens-16k.md) (16k baseline), [dev-009](../docs/log/runs/dev-009-max-tokens-32k.md) (32k ablation), [dev-010-bf](../docs/log/runs/dev-010-bf-budget-forcing.md) (FF budget forcing), [dev-013-verify](../docs/log/runs/dev-013-verify-holdout-20p.md) (verify_prompt, holdout_20p), [dev-014-precision](../docs/log/runs/dev-014-precision-holdout-20p.md) (exact-form prompt, holdout_20p).

In [29]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

_q_by_id = {d["id"]: d["question"] for d in data}
multi_free  = [r for r in free_res if n_ans_placeholders(_q_by_id[r["id"]]) > 1]
single_free = [r for r in free_res if r not in multi_free]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

mcq_pct   = acc(mcq_res)
free_pct  = acc(free_res)
multi_pct = acc(multi_free)
overall_pct = acc(results)
n_total = len(results)

print("=" * 50)
print("DEV SET RESULTS")
print("=" * 50)
print(f"  RUN_ID       : {RUN_ID}")
print(f"  MAX_TOKENS   : {MAX_TOKENS}")
print(f"  PROMPT       : {PROMPT_VARIANT}")
print(f"  XML_WRAP     : {USE_XML_WRAPPERS}")
print(f"  BUDGET_FORCE : {BUDGET_FORCING}")
print(f"  SC           : {SELF_CONSISTENCY}" + (f" (K={SC_K})" if SELF_CONSISTENCY else ""))
print(f"  SMOKE        : {SMOKE_TEST}")
print(f"  N            : {n_total}  ({len(mcq_res)} MCQ, {len(free_res)} free-form)")
print("-" * 50)
if mcq_res:
    print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({mcq_pct:.2f}%)")
if free_res:
    print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({free_pct:.2f}%)")
if multi_free:
    print(f"  Multi-blank: {sum(r['correct'] for r in multi_free):4d} / {len(multi_free):4d}  ({multi_pct:.2f}%)")
if single_free:
    print(f"  Single-blank: {sum(r['correct'] for r in single_free):4d} / {len(single_free):4d}  ({acc(single_free):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {n_total:4d}  ({overall_pct:.2f}%)")
print("=" * 50)

print(f"\nArtifact  : {Path(OUTPUT_PATH).name}")
print(f"Registry  : docs/log/runs/{RUN_ID}-*.md")

DEV SET RESULTS
  RUN_ID       : dev-precision-prompt
  MAX_TOKENS   : 16384
  PROMPT       : precision_xml_verify
  XML_WRAP     : False
  BUDGET_FORCE : False
  SC           : False
  SMOKE        : False
  N            : 225  (75 MCQ, 150 free-form)
--------------------------------------------------
  MCQ        :   60 /   75  (80.00%)
  Free-form  :   93 /  150  (62.00%)
  Multi-blank:   46 /   82  (56.10%)
  Single-blank:   47 /   68  (69.12%)
  Overall    :  153 /  225  (68.00%)

Artifact  : dev_results_precision_xml_verify_16k.jsonl
Registry  : docs/log/runs/dev-precision-prompt-*.md


## 11. Save results

Same schema as the starter (`id`, `is_mcq`, `gold`, `response`, `correct`). Uses Drive when `DRIVE_PROJECT_ROOT` exists.

In [30]:
SAVE_EVAL = True

try:
    out_path = DRIVE_PROJECT_ROOT / "results" / Path(OUTPUT_PATH).name
except NameError:
    out_path = Path(OUTPUT_PATH)

out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path.resolve()}")

Saved 225 records to /content/drive/MyDrive/CSE151B/results/dev_results_precision_xml_verify_16k.jsonl


In [31]:
try:
    from google.colab import runtime
    runtime.unassign()
except ImportError:
    print("Not running in Colab — skipping runtime termination.")